# Data Setup

## Import packages and CSV files

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# constants
SEASON_START = 2014
SEASON_END = 2025

# path to data directory
data_directory = Path("../data")

# opening CSV files
circuits = pd.read_csv(data_directory / "circuits.csv", na_values=[r"\N"])
constructor_standings = pd.read_csv(data_directory / "constructor_standings.csv", na_values=[r"\N"])
constructors = pd.read_csv(data_directory / "constructors.csv", na_values=[r"\N"])
driver_standings = pd.read_csv(data_directory / "driver_standings.csv", na_values=[r"\N"])
drivers = pd.read_csv(data_directory / "drivers.csv", na_values=[r"\N"])
qualifying = pd.read_csv(data_directory / "qualifying.csv", na_values=[r"\N"])
races = pd.read_csv(data_directory / "races.csv", na_values=[r"\N"])
results = pd.read_csv(data_directory / "results.csv", na_values=[r"\N"])
seasons = pd.read_csv(data_directory / "seasons.csv", na_values=[r"\N"])

## Helper methods

In [2]:
print(races.columns)

# parses lap time string to timedelta
def parse_lap_time(t):
    if pd.isna(t) or t in (0, "0", "\\N", ""):
        return pd.NaT
    t_str = str(t).strip()
    if t_str.count(":") == 1:
        t_str = "00:" + t_str
    return pd.to_timedelta(t_str, errors="coerce")

Index(['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url',
       'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time',
       'quali_date', 'quali_time', 'sprint_date', 'sprint_time'],
      dtype='object')


## Data pipeline

In [ ]:
# iterate over each race
for race in races.itertuples():
    # race information
    race_id = int(race.raceId)
    season = race.year
    
    if int(season) < SEASON_START or int(season) > 2014:
        continue
    
    round = int(race.round)
    circuit_id = int(race.circuitId)
    circuit_ref = circuits.loc[circuits["circuitId"] == circuit_id, "circuitRef"].item()
    circuit_type = circuits.loc[circuits["circuitId"] == circuit_id, "circuitType"].item()
    
    # get previous race id
    if round > 1:
        prev_race_id = int(races.loc[(races["year"] == season) & (races["round"] == (round - 1)), "raceId"].item())
    else:
        prev_race_id = -1
    
    # get qualifying times
    race_quali = qualifying.loc[qualifying["raceId"] == race_id].copy()
    for col in ["q1", "q2", "q3"]:
        race_quali[col] = race_quali[col].apply(parse_lap_time)
    
    # get fastest qualifying time in each session
    fastest_in_session = {
        "q3": race_quali["q3"].min(),
        "q2": race_quali["q2"].min(),
        "q1": race_quali["q1"].min(),
    }
    
    # iterate over each driver in the race
    for driver in race_quali.itertuples():
        # driver information
        driver_id = int(driver.driverId)
        constructor_id = int(driver.constructorId)

        if pd.notna(driver.q3):
            session, qualifying_time = 3, driver.q3
        elif pd.notna(driver.q2):
            session, qualifying_time = 2, driver.q2
        elif pd.notna(driver.q1):
            session, qualifying_time = 1, driver.q1
        else:
            session, qualifying_time = 0, pd.NaT

        # calculate gap to fastest time in last session
        if session != 0:
            gap_td = qualifying_time - fastest_in_session[f"q{session}"]
            gap_seconds = gap_td.total_seconds()
        else:
            gap_seconds = 0
        
        # get driver race result
        driver_race_result = results.loc[(results["raceId"] == race_id) & (results["driverId"] == driver_id), ["grid", "positionOrder", "positionText"]].iloc[0]
        
        starting_pos = int(driver_race_result["grid"])
        ending_pos = int(driver_race_result["positionOrder"])
        dnf = 1 if driver_race_result["positionText"].isalpha() else 0
        
        # get driver and constructor championship information
        driver_champ_points = 0
        driver_champ_pos = 0
        driver_wins = 0
        constructor_champ_points = 0
        constructor_champ_pos = 0
        constructor_wins = 0
        
        if prev_race_id > 0:
            driver_standing = driver_standings.loc[(driver_standings["raceId"] == prev_race_id) & (driver_standings["driverId"] == driver_id), ["points", "positionText", "wins"]]
            constructor_standing = constructor_standings.loc[(constructor_standings["raceId"] == prev_race_id) & (constructor_standings["constructorId"] == constructor_id), ["points", "positionText", "wins"]]
            
            if not driver_standing.empty:
                driver_standing = driver_standing.iloc[0]
                driver_champ_points = int(driver_standing["points"])
                driver_champ_pos = int(driver_standing["positionText"])
                driver_wins = int(driver_standing["wins"])
            
            if not constructor_standing.empty:
                constructor_standing = constructor_standing.iloc[0]
                constructor_champ_points = int(constructor_standing["points"])
                constructor_champ_pos = int(constructor_standing["positionText"])
                constructor_wins = int(constructor_standing["wins"])